# Buckaroo + xorq

`XorqBuckarooWidget` and `XorqBuckarooInfiniteWidget` let you point Buckaroo at an [xorq](https://github.com/letsql/xorq) (ibis-style) lazy expression instead of a materialized pandas DataFrame.

What you get:

- **`XorqBuckarooWidget`** — runs the standard analysis pipeline against a sampled head of the expression. Same UI as the pandas widget.
- **`XorqBuckarooInfiniteWidget`** — never materializes the full expression. Each scroll position pushes a `LIMIT/OFFSET` to the backend and streams the resulting Arrow window straight to the browser as Parquet bytes (no pandas detour).

This notebook demonstrates both, against a real parquet file (citibike, 100k rows) and a synthetic NumPy-generated frame (1.2M rows).

In [1]:
import numpy as np
import pandas as pd
import xorq.api as xo

from buckaroo.xorq_buckaroo import (
    XorqBuckarooWidget,
    XorqBuckarooInfiniteWidget,
)

Buckaroo has been enabled as the default DataFrame viewer.  To return to default dataframe visualization use `from buckaroo import disable; disable()`


## 1. Citibike — real parquet through xorq

Load the parquet, wrap it in an xorq `memtable`, and hand the lazy expression to the widget. Buckaroo's analysis pipeline runs against the xorq backend (DataFusion under the hood) — no pandas computation in the hot path.

In [2]:
citibike_df = pd.read_parquet("./citibike-trips-2016-04.parq")
citibike_expr = xo.memtable(citibike_df)
print(f"rows: {citibike_expr.count().execute():,}")
print(f"columns: {len(citibike_expr.schema())}")

rows: 100,000
columns: 15


In [3]:
XorqBuckarooWidget(citibike_expr)

XorqBuckarooWidget(buckaroo_options={'sampled': ['random'], 'auto_clean': ['aggressive', 'conservative'], 'pos…

### Same data, infinite-scroll variant

The infinite widget keeps the expression lazy. Drag the scrollbar — each window is fetched on demand via `expr.limit(n, offset=k).to_pyarrow()`.

In [4]:
XorqBuckarooInfiniteWidget(citibike_expr)

XorqBuckarooInfiniteWidget(buckaroo_options={'sampled': ['random'], 'auto_clean': ['aggressive', 'conservative…

### Lazy postprocessing

Because the backend stays lazy, a postprocessing step is just another expression node — it composes with the windowed query rather than running ahead of it. Filtering to long trips below adds a `WHERE` clause that gets pushed down with the `LIMIT/OFFSET`.

In [ ]:
long_trips = citibike_expr.filter(citibike_expr.tripduration > 600)
XorqBuckarooInfiniteWidget(long_trips)

## 2. 1.2M-row synthetic frame

Generate a frame larger than anything you'd want to render at once, hand it to the infinite widget, and confirm scrolling stays responsive. The widget never holds more than one window's worth of rows at a time on the JS side.

In [5]:
N = 1_200_000
rng = np.random.default_rng(42)

big_df = pd.DataFrame({
    "id": np.arange(N, dtype="int64"),
    "category": rng.choice(["alpha", "beta", "gamma", "delta"], size=N),
    "value": rng.normal(100, 15, size=N),
    "score": rng.uniform(0, 1, size=N),
    "count": rng.integers(0, 1000, size=N),
})
big_expr = xo.memtable(big_df)
print(f"rows: {big_expr.count().execute():,}")

rows: 1,200,000


In [6]:
XorqBuckarooInfiniteWidget(big_expr)

XorqBuckarooInfiniteWidget(buckaroo_options={'sampled': ['random'], 'auto_clean': ['aggressive', 'conservative…